<img src="https://courseware.decoded.com/images/decoded/logo-decoded.png" align="left" width="100px"/>

# Day 3 — Backup: generic test, snapshot, incremental, unit test

<small>Data Creators › Day 3 — instructor backup</small>

---
<small>&copy; 2026 Decoded Limited. All rights reserved. [decoded.com](https://decoded.com)</small>

Instructor only: show this if the room asks about reusable tests, history, large tables or testing logic. It runs on the price sheet in `courseware/day3-dbt/demos/price_sheet`, after the anchor's section 3 cells (the four models and the `to_usd` macro must be built).

The restated FX rate stays in the warehouse until the next `reset.sh` or `dbt seed`.

In [ ]:
# Setup: the demo environment on PATH, then work in the price sheet folder
import os
from pathlib import Path
VENV = Path(os.environ.get("CG_DXP_VENV", Path.home() / ".venvs" / "cg-dxp"))
os.environ["PATH"] = ":".join([str(VENV / "bin"), "/opt/homebrew/bin", os.environ["PATH"]])
os.environ.setdefault("DBT_SEND_ANONYMOUS_USAGE_STATS", "false")
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "courseware" / "day3-dbt").is_dir())
PS = ROOT / "courseware/day3-dbt/demos/price_sheet"
os.chdir(PS)
(PS / "snapshots").mkdir(exist_ok=True); (PS / "tests" / "generic").mkdir(parents=True, exist_ok=True)
print("Working in", PS)

#### A generic test of your own: this morning's rule, for any column

In [ ]:
%%writefile tests/generic/test_positive.sql
{# positive: this morning's rule as a generic test, so any column can use it. dbt fills in the model and the column. #}
{% test positive(model, column_name) %}
select * from {{ model }} where {{ column_name }} <= 0
{% endtest %}

In [ ]:
%%writefile models/marts/schema.yml
# schema.yml: what the model and its columns mean, and the promises dbt checks
version: 2

models:
  - name: fct_prices_usd
    description: "End-of-day prices in US dollars."
    columns:
      - name: instrument_id
        description: "ISIN: the instrument the price is for."
        tests:
          - not_null
      - name: price_usd
        description: "Close price times the day's FX rate to USD."
        tests:
          - not_null
          - positive                    # our own generic test, used like not_null

In [ ]:
!dbt test --select fct_prices_usd
# Four tests on the mart now: the two not_null tests, this morning's test of your own, and the new positive test, used in the YAML like a built-in one.

In [ ]:
!cat target/compiled/price_sheet/models/marts/schema.yml/positive_fct_prices_usd_price_usd.sql
# dbt filled in the model and the column: the same select-the-violations query as this morning's test, now reusable.

#### A snapshot: keep the old versions of rows that change

<img src="../../courseware/common/diagrams/png/d3-snapshot.png" alt="A snapshot keeps every version of a row; the raw table keeps only the latest" style="max-height:360px; max-width:100%;">

*A snapshot keeps every version of a row; the raw table keeps only the latest*

<details><summary>Notes</summary>

- The vendor restates a rate: the raw table is overwritten, and yesterday's value is gone.
- The snapshot table keeps both: the old version closed (dbt_valid_to set), the new one current (dbt_valid_to empty).
- So you can answer: what did we think the rate was on Monday, and what do we think it is now?

</details>

In [ ]:
%%writefile snapshots/fx_rates_snapshot.yml
# A snapshot: dbt keeps every version of each FX rate, so a restated rate does not overwrite the old one
snapshots:
  - name: fx_rates_snapshot
    relation: source('fx_feed', 'fx_rates_raw')   # what to keep the history of
    config:
      unique_key: [from_ccy, rate_dt]           # what makes one row: a currency on a day
      strategy: check                           # a new version when a column below changes
      check_cols: [rate]

<img src="../../courseware/common/diagrams/png/d3-snapshot-run.png" alt="What each dbt snapshot run does, row by row" style="max-height:300px; max-width:100%;">

*What each dbt snapshot run does, row by row*

<details><summary>Notes</summary>

- `unique_key` says which rows are the same thing: one currency on one day.
- `check_cols` says which change counts: here, a new rate. Any other column can change without a new version.
- Run it on a schedule, after each delivery: dbt can only keep the versions it sees.

</details>

In [ ]:
!dbt snapshot
# First run: one version of every FX rate, each one open (dbt_valid_to is empty).

In [ ]:
# The vendor restates one rate: the GBP close for 5 November was wrong. The raw table now holds only the new value.
import duckdb
con = duckdb.connect("warehouse/price_sheet.duckdb")
con.execute("update demo.fx_rates_raw set rate = 1.300000 where from_ccy = 'GBP' and rate_dt = '2026-11-05'")
con.close()

In [ ]:
!dbt snapshot
# Second run: dbt sees the rate changed, closes the old version and opens a new one.

In [ ]:
!dbt show --inline "select from_ccy, rate_dt, rate, dbt_valid_from, dbt_valid_to from demo.fx_rates_snapshot where from_ccy = 'GBP' and rate_dt = '2026-11-05' order by dbt_valid_from" --limit 10
# Two versions of one rate: the old one closed (dbt_valid_to is set), the new one open. The raw table kept only the new one.

#### An incremental model: only the new rows

In [ ]:
%%writefile models/marts/fct_prices_incremental.sql
-- fct_prices_incremental: the first run loads every day; later runs add only the days the table does not have
{{ config(materialized='incremental') }}

select * from {{ ref('int_prices_usd') }}
{% if is_incremental() %}
where price_date > (select max(price_date) from {{ this }})   -- {{ this }} is the table as it is now
{% endif %}

In [ ]:
!dbt run --select fct_prices_incremental
# First run: the table does not exist yet, so dbt builds it from every row.

In [ ]:
!dbt run --select fct_prices_incremental
# Second run: the table exists, so is_incremental() is true and dbt adds only the days the table does not have. Today there are none.

In [ ]:
!cat target/compiled/price_sheet/models/marts/fct_prices_incremental.sql
# The second run's SELECT: the where clause is there now. dbt inserts these rows instead of rebuilding the table.

#### A unit test: the model's logic on made-up rows

[dbt docs: unit tests](https://docs.getdbt.com/docs/build/unit-tests)

In [ ]:
%%writefile models/intermediate/unit_tests.yml
# A unit test: run int_prices_usd on rows we make up, and compare with the rows we expect
unit_tests:
  - name: int_prices_usd_converts_to_dollars
    model: int_prices_usd
    given:                                      # made-up input rows, in place of the real models
      - input: ref('stg_prices')
        rows:
          - {instrument_id: 'A', price_date: '2026-11-05', native_ccy: 'USD', price_native: 100}
          - {instrument_id: 'B', price_date: '2026-11-05', native_ccy: 'GBP', price_native: 10}
      - input: ref('stg_fx_rates')
        rows:
          - {from_ccy: 'GBP', rate_date: '2026-11-05', fx_rate: 1.25}
    expect:                                     # what the model should return: USD unchanged, 10 GBP x 1.25
      rows:
        - {instrument_id: 'A', price_usd: 100}
        - {instrument_id: 'B', price_usd: 12.5}

In [ ]:
!dbt test --select test_type:unit
# A unit test runs the model SQL on made-up rows and compares with expected rows: no warehouse data, pure logic. Change 12.5 to 12 and run it again to see it fail.

<div style="border:2px solid #434343; border-radius:8px; margin:10px 0;">
<div style="background:#D0E0E3; color:#1a1a1a; padding:8px 14px; font-weight:bold;">🤖 PROMPT</div>
<div style="background:#fff; padding:12px 14px; color:#1a1a1a;"><span style="display:inline-block;margin:0 6px 4px 0;border:1px solid #434343;border-radius:4px;overflow:hidden;font-size:0.85em;line-height:1.7;white-space:nowrap;"><span style="background:#D0E0E3;color:#1a1a1a;padding:2px 7px;font-weight:700;letter-spacing:0.05em;">LEARNERS</span><span style="background:#fff;color:#1a1a1a;padding:2px 7px;">Copilot Chat · Chat view, in the learner project</span></span><pre style="white-space:pre-wrap;background:#f8f8f8;padding:8px;border-radius:4px;">/ai-catalyst adding-dbt-unit-test
Add a dbt unit test for stg_holdings covering: a null ISIN becomes CASH_&lt;ccy&gt;; fund codes are trimmed and upper-cased; a row with a null position date is dropped.</pre><span style="display:inline-block;margin:0 6px 4px 0;border:1px solid #434343;border-radius:4px;overflow:hidden;font-size:0.85em;line-height:1.7;white-space:nowrap;"><span style="background:#C9DAF8;color:#1a1a1a;padding:2px 7px;font-weight:700;letter-spacing:0.05em;">INSTRUCTOR</span><span style="background:#fff;color:#1a1a1a;padding:2px 7px;">Claude Code</span></span><pre style="white-space:pre-wrap;background:#f8f8f8;padding:8px;border-radius:4px;">/adding-dbt-unit-test  (same text) — or simply: add a unit_tests entry for stg_holdings in a YAML file covering those three cases</pre></div></div>

#### Tidy up

The generic test, the snapshot, the incremental model and the unit test stay in the price sheet project until the next `bash reset.sh`, so a later `dbt build` builds them too. Run the reset before the next delivery.